In [1]:
# optimizes these parameters using gradient descent

## Prerequisite Code

In [23]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

## Hyperparameters
### Hyperparameters are adjustable parameters that let you control the model optimization process.

In [7]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

## Optimization Loop
### Once we set our hyperparameters, we can then train and optimize our model with an optimization loop. Each iteration of the optimization loop is called an epoch.

## Loss Function

### Common loss functions include nn.MSELoss (Mean Square Error) for regression tasks, and nn.NLLLoss (Negative Log Likelihood) for classification. nn.CrossEntropyLoss combines nn.LogSoftmax and nn.NLLLoss.

In [13]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

## Optimizer
### Optimization is the process of adjusting model parameters to reduce model error in each training step.

In [16]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Full Implementation

In [19]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [21]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.307570  [   64/60000]
loss: 2.294706  [ 6464/60000]
loss: 2.281291  [12864/60000]
loss: 2.271249  [19264/60000]
loss: 2.245659  [25664/60000]
loss: 2.223624  [32064/60000]
loss: 2.222952  [38464/60000]
loss: 2.194830  [44864/60000]
loss: 2.193604  [51264/60000]
loss: 2.150525  [57664/60000]
Test Error: 
 Accuracy: 47.5%, Avg loss: 2.153560 

Epoch 2
-------------------------------
loss: 2.170106  [   64/60000]
loss: 2.157210  [ 6464/60000]
loss: 2.108506  [12864/60000]
loss: 2.117072  [19264/60000]
loss: 2.057738  [25664/60000]
loss: 2.009259  [32064/60000]
loss: 2.026565  [38464/60000]
loss: 1.959241  [44864/60000]
loss: 1.963189  [51264/60000]
loss: 1.870680  [57664/60000]
Test Error: 
 Accuracy: 57.6%, Avg loss: 1.884000 

Epoch 3
-------------------------------
loss: 1.925589  [   64/60000]
loss: 1.886477  [ 6464/60000]
loss: 1.783463  [12864/60000]
loss: 1.810075  [19264/60000]
loss: 1.694064  [25664/60000]
loss: 1.657820  [32064/600